# Planning the initial history load

The daily capture only sees today. Backtests need the past, so step 2 of phase 1 loads history once. This notebook decides **what** to load, **how much** it is, **what it costs** against the sources' limits, and **what has to be true** of the loader, by downloading samples and looking at them.

| Question | Why it matters for the loader |
| --- | --- |
| Which historical files exist, back to when? | The list of items to fetch |
| Are old files the same format as today's? | What the parsers must handle |
| Do old files change after publication? | How often history must be re-captured |
| How expensive is weather history? | Pacing, chunk size, how many days the load takes |
| Which weather runs were really available at decision time? | Which exact runs to fetch for the last months |

Claims are labelled as in the first notebook: **Observed** (a cell shows it) or **Source** (quoted from the publisher, with a link). Requests are paced 2 s apart and cached per day under `data/probe/`, so re-running costs nothing.

## 0. Setup

In [1]:
import io
import json
import re
import time
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import httpx
import pandas as pd

from jtf.ingest.fetch import USER_AGENT
from jtf.ingest.sources import (
    JEPX_DOWNLOAD,
    JEPX_SPOT_PAGE,
    JST,
    WEATHER_MODELS,
    WEATHER_POINTS,
    WEATHER_VARIABLES,
    FetchItem,
)

now_jst = datetime.now(JST)
CACHE = Path("../data/probe") / f"{now_jst:%Y-%m-%d}"
CACHE.mkdir(parents=True, exist_ok=True)
client = httpx.Client(timeout=60, follow_redirects=True, headers={"User-Agent": USER_AGENT})


def get(item: FetchItem) -> tuple[bytes, dict]:
    """Download once per day and keep the headers; uncached requests are spaced 2 s apart."""
    body_path = CACHE / f"{item.source}__{item.filename}"
    head_path = body_path.parent / (body_path.name + ".headers.json")
    if not body_path.exists():
        time.sleep(2)
        r = client.request(
            item.method,
            item.url,
            params=item.params or None,
            data=item.form or None,
            headers=item.headers or None,
        )
        head = {k.lower(): v for k, v in r.headers.items() if k.lower() != "set-cookie"}
        head["x-status"] = r.status_code
        head["x-fetched-at"] = datetime.now(UTC).isoformat()
        body_path.write_bytes(r.content)
        head_path.write_text(json.dumps(head))
    return body_path.read_bytes(), json.loads(head_path.read_text())


print(f"probing at {now_jst:%Y-%m-%d %H:%M} JST")

probing at 2026-11-15 14:46 JST


## 1. TEPCO area actuals: what history exists

**Step: list the files.** TEPCO has two download pages: [monthly files from February 2024](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html) and [yearly files for older fiscal years](https://www.tepco.co.jp/forecast/html/area_jukyu_p-j.html). Read both and collect every CSV link.

In [2]:
PAGES = {
    "monthly": "https://www.tepco.co.jp/forecast/html/area_jukyu-j.html",
    "yearly": "https://www.tepco.co.jp/forecast/html/area_jukyu_p-j.html",
}
rows = []
for kind, url in PAGES.items():
    html, _ = get(FetchItem("tepco_pages", f"{kind}.html", url))
    for path in sorted(set(re.findall(r"/forecast/html/images/[\w-]+\.csv", html.decode("utf-8")))):
        rows.append(
            {"page": kind, "file": path.rsplit("/", 1)[-1], "url": "https://www.tepco.co.jp" + path}
        )
tepco_files = pd.DataFrame(rows)
tepco_files.groupby("page")["file"].agg(["count", "min", "max"])

,count,min,max
page,,,
monthly,34,eria_jukyu_202402_03.csv,eria_jukyu_202611_03.csv
yearly,8,area-2016.csv,area-2023.csv


**Observed:** 8 yearly files (FY2016–FY2023) and one monthly file per month from February 2024 to the current month.

**Step: sample files across the whole range** and look at encoding, size, the unit row and, through `Last-Modified`, when each file was last rewritten.

In [3]:
def decode_tepco(raw: bytes) -> tuple[str, str]:
    """TEPCO files are UTF-8 or CP932 (Shift_JIS with Microsoft's extensions), file by file."""
    try:
        return raw.decode("utf-8-sig"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("cp932"), "cp932"


def tepco_csv(name: str) -> tuple[list[str], dict]:
    url = tepco_files.loc[tepco_files["file"] == name, "url"].iloc[0]
    raw, headers = get(FetchItem("tepco_actuals_history", name, url))
    text, encoding = decode_tepco(raw)
    headers["x-encoding"] = encoding
    return text.splitlines(), headers


sample = [
    "area-2016.csv",
    "area-2023.csv",
    "eria_jukyu_202402_03.csv",
    "eria_jukyu_202408_03.csv",
    "eria_jukyu_202502_03.csv",
    "eria_jukyu_202508_03.csv",
    "eria_jukyu_202602_03.csv",
    "eria_jukyu_202610_03.csv",
    f"eria_jukyu_{now_jst:%Y%m}_03.csv",
]
rows = []
for name in sample:
    lines, headers = tepco_csv(name)
    rows.append(
        {
            "file": name,
            "encoding": headers["x-encoding"],
            "bytes": int(headers.get("content-length", 0)),
            "last modified": pd.to_datetime(headers.get("last-modified")).tz_convert("Asia/Tokyo"),
            "unit row": lines[0][:20],
        }
    )
pd.DataFrame(rows)

,file,encoding,bytes,last modified,unit row
0,area-2016.csv,cp932,513095,2021-12-14 09:59:50+09:00,"単位[万kWh],,,,,,,,,,,,"
1,area-2023.csv,cp932,434359,2024-10-23 21:15:05+09:00,単位[万kWh]
2,eria_jukyu_202402_03.csv,utf-8,119968,2024-10-24 10:25:03+09:00,"単位[MW平均],,,供給力"
3,eria_jukyu_202408_03.csv,utf-8,128762,2024-10-24 10:25:02+09:00,"単位[MW平均],,,供給力"
4,eria_jukyu_202502_03.csv,utf-8,116474,2025-03-29 05:15:02+09:00,"単位[MW平均],,,供給力"
5,eria_jukyu_202508_03.csv,utf-8,129620,2025-09-25 10:45:00+09:00,"単位[MW平均],,,供給力"
6,eria_jukyu_202602_03.csv,utf-8,116020,2026-03-11 14:50:06+09:00,"単位[MW平均],,,供給力"
7,eria_jukyu_202610_03.csv,cp932,136134,2026-11-01 00:10:05+09:00,"単位[MW平均],,,供給力"
8,eria_jukyu_202611_03.csv,cp932,63405,2026-11-15 13:05:01+09:00,"単位[MW平均],,,供給力"


**What it shows.**

- **Observed: the encoding changes between files of the same series.** The monthly files were UTF-8 for a long stretch, the newest ones are CP932, and the yearly files are CP932. Decoding as Shift_JIS, as the first notebook did with the current month, fails on the older monthly files
- **Observed: two different units.** Yearly files are in 万kWh per hour, monthly files in MW averaged over the slot
- **Observed: files are rewritten after the month ends.** A month's file was last modified weeks into the following month (for example, February 2025 at the end of March 2025, August 2025 in late September). In October 2024 TEPCO rewrote older files in bulk: the February and August 2024 files and the FY2023 yearly file all carry the same October 2024 date
- **Source**, the [actuals page](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html): 「公表値は、過去にさかのぼり修正させていただく場合があります。」 Published values may be corrected retroactively. The dates above show it happening, and how late

**Step: load both formats.** The yearly files need their own reading: a header spanning two rows and, in older years, numbers with thousands separators.

In [4]:
def load_tepco(lines: list[str]) -> tuple[pd.DataFrame, str]:
    """Both TEPCO formats: one header row (monthly) or a header spanning two rows (yearly)."""
    unit = lines[0].split(",")[0]
    header = next(i for i, line in enumerate(lines) if line.startswith("DATE"))
    names = lines[header].split(",")
    body = lines[header + 1 :]
    if body[0].startswith(",,,"):
        names = names[:3] + body[0].split(",")[3:]
        body = body[1:]
    df = pd.read_csv(io.StringIO("\n".join(body)), names=names, thousands=",")
    df = df.rename(columns={"東京エリア需要": "エリア需要"}).dropna(subset=["エリア需要"])
    df["start"] = pd.to_datetime(df["DATE"] + " " + df["TIME"], format="%Y/%m/%d %H:%M")
    return df, unit


print("yearly FY2016 header rows:", *tepco_csv("area-2016.csv")[0][:4], sep="\n  ")
yearly, yearly_unit = load_tepco(tepco_csv("area-2023.csv")[0])
monthly, monthly_unit = load_tepco(tepco_csv("eria_jukyu_202402_03.csv")[0])
for name, df, unit in (
    ("yearly FY2023", yearly, yearly_unit),
    ("monthly 2024-02", monthly, monthly_unit),
):
    step = df["start"].diff().mode().iloc[0]
    print(
        f"{name}: {unit}, {df['start'].min()} to {df['start'].max()}, {len(df)} rows, step {step}"
    )
print("\ncolumns only in the monthly file:", sorted(set(monthly.columns) - set(yearly.columns)))
print("columns only in the yearly file:", sorted(set(yearly.columns) - set(monthly.columns)))

yearly FY2016 header rows:
  単位[万kWh],,,,,,,,,,,,,,
  DATE,TIME,東京エリア需要,供給力,,,,,,,,,,,
  ,,,原子力,火力,水力,地熱,バイオマス,太陽光発電実績,太陽光出力制御量,風力発電実績,風力出力制御量,揚水,連系線,合計
  2016/4/1,0:00,"2,555",0,2258,92,0,2,0,0,2,0,0,201,"2,555"
yearly FY2023: 単位[万kWh], 2023-04-01 00:00:00 to 2024-01-31 23:00:00, 7344 rows, step 0 days 01:00:00
monthly 2024-02: 単位[MW平均], 2024-02-01 00:00:00 to 2024-02-29 23:30:00, 1392 rows, step 0 days 00:30:00

columns only in the monthly file: ['その他', '火力(LNG)', '火力(その他)', '火力(石油)', '火力(石炭)', '蓄電池']
columns only in the yearly file: ['火力']


**What it shows.** **Observed:**

- The yearly format's header spans two rows (`DATE,TIME,東京エリア需要,供給力`, then `,,,原子力,火力,…`), and FY2016 writes numbers like `"2,555"`
- Thermal power is one `火力` column in the yearly files, split into LNG, coal, oil and other in the monthly files, which also add batteries (`蓄電池`) and `その他`
- **The yearly FY2023 file stops on 31 January 2024**, exactly where the monthly files begin: the two series meet end to end, with no overlap

**Step: check the join.** Convert the yearly values to MW (万kWh in one hour is 10 MWh, so 10 MW on average) and look across the boundary.

In [5]:
before = yearly.set_index("start").loc["2024-01-31 18:00":, ["エリア需要", "太陽光発電実績"]] * 10
after = monthly.set_index("start").loc[:"2024-02-01 02:00", ["エリア需要", "太陽光発電実績"]]
boundary = pd.concat([before.assign(file="yearly, 万kWh x 10"), after.assign(file="monthly, MW")])
boundary.rename(columns={"エリア需要": "demand MW", "太陽光発電実績": "solar MW"})

,demand MW,solar MW,file
start,,,
2024-01-31 18:00:00,38330,0,"yearly, 万kWh x 10"
2024-01-31 19:00:00,37580,0,"yearly, 万kWh x 10"
2024-01-31 20:00:00,36440,0,"yearly, 万kWh x 10"
2024-01-31 21:00:00,34560,0,"yearly, 万kWh x 10"
2024-01-31 22:00:00,32570,0,"yearly, 万kWh x 10"
2024-01-31 23:00:00,30420,0,"yearly, 万kWh x 10"
2024-02-01 00:00:00,28755,0,"monthly, MW"
2024-02-01 00:30:00,27625,0,"monthly, MW"
2024-02-01 01:00:00,27195,0,"monthly, MW"


In [6]:
daily = pd.concat(
    [
        (yearly.set_index("start")["太陽光発電実績"] * 10).loc["2024-01-25":].resample("D").mean(),
        monthly.set_index("start")["太陽光発電実績"].loc[:"2024-02-07"].resample("D").mean(),
    ]
)
daily.rename("daily mean solar MW").round(0).to_frame().T

start,2024-01-25,2024-01-26,2024-01-27,2024-01-28,2024-01-29,2024-01-30,2024-01-31,2024-02-01,2024-02-02,2024-02-03,2024-02-04,2024-02-05,2024-02-06,2024-02-07
daily mean solar MW,3065.0,3135.0,3056.0,2242.0,3204.0,3207.0,2952.0,2778.0,1565.0,3324.0,1672.0,336.0,345.0,3114.0


**What it shows.** **Observed:** demand runs on smoothly across midnight and daily solar stays at the same level, so ×10 is the right conversion. What the boundary can't show is whether an hourly row is labelled by the hour's start or its end; that only matters if the hourly years are ever used.

> **Learned → design**
>
> - **History is 34+ monthly files and 8 yearly files**, generated from known name patterns: `eria_jukyu_YYYYMM_03.csv` from 2024-02, `area-YYYY.csv` for FY2016–2023. About 8 MB in all
> - **Parsers detect the encoding per file** (UTF-8 first, then CP932), never assume it
> - **The yearly format gets its own parser**: two-row header, thousands separators, 万kWh × 10 → MW, one thermal column. Phase 1 trains on the 30-minute data from February 2024; the hourly years are loaded raw and parsed later
> - **Corrections land weeks after a month ends, and sometimes in bulk months later.** The daily capture's "previous month on days 1–5" is too short: it should fetch the previous month every day. A monthly re-capture of every monthly file catches the rare bulk rewrites, for about 4 MB a month. Each capture is a version, so this is also the raw material for the optimism-gap experiment

## 2. JEPX prices: what history exists

**Step:** request one recent and two old fiscal-year files through the same form POST the [spot market page](https://www.jepx.jp/electricpower/market-data/spot/) uses, and compare their shape.

In [7]:
def jepx_year(fy: int) -> tuple[pd.DataFrame, dict, int]:
    name = f"spot_summary_{fy}.csv"
    raw, headers = get(
        FetchItem(
            "jepx_spot_history",
            name,
            JEPX_DOWNLOAD,
            method="POST",
            form={"dir": "spot_summary", "file": name},
            headers={"Referer": JEPX_SPOT_PAGE},
        )
    )
    if headers.get("x-status") != 200 or not raw:
        return pd.DataFrame(), headers, len(raw)
    return pd.read_csv(io.BytesIO(raw), encoding="shift_jis"), headers, len(raw)


fy_now = now_jst.year if now_jst.month >= 4 else now_jst.year - 1
jepx = {fy: jepx_year(fy) for fy in (2015, 2016, fy_now)}
for fy, (df, headers, size) in jepx.items():
    days = df["受渡日"].nunique() if len(df) else 0
    first = df["受渡日"].iloc[0] if len(df) else "-"
    print(f"FY{fy}: status {headers.get('x-status')}, {size:,} bytes, {days} days from {first}")
old, new = jepx[2016][0], jepx[fy_now][0]
print("\nsame columns in FY2016 and today:", list(old.columns) == list(new.columns))

FY2015: status 200, 1,905,864 bytes, 366 days from 2015/04/01
FY2016: status 200, 1,898,748 bytes, 365 days from 2016/04/01
FY2026: status 200, 1,450,151 bytes, 230 days from 2026/04/01

same columns in FY2016 and today: True


In [8]:
print("slots per day in FY2016:", sorted(old.groupby("受渡日").size().unique()))
print("missing Tokyo prices in FY2016:", old["エリアプライス東京(円/kWh)"].isna().sum())
old[["受渡日", "時刻コード", "システムプライス(円/kWh)", "エリアプライス東京(円/kWh)"]].head(3)

slots per day in FY2016: [np.int64(48)]
missing Tokyo prices in FY2016: 0


,受渡日,時刻コード,システムプライス(円/kWh),エリアプライス東京(円/kWh)
0,2016/04/01,1,6.61,6.69
1,2016/04/01,2,6.34,6.34
2,2016/04/01,3,6.34,6.34


**What it shows.** **Observed:** FY2015 and FY2016 exist; FY2016 has exactly the same columns as today's file, 48 slots every day and no missing Tokyo prices. A year is about 1.9 MB.

> **Learned → design**
>
> - **Past fiscal years load once**: FY2016 to last year, about 10 files and 20 MB. The current year keeps coming from the daily 11:00 capture
> - **One parser for every year.** Nothing in the format changed
> - **`known_at` for history is 10:30 on the day before delivery**, the conservative rule from the as-of table

## 3. Open-Meteo Previous Runs: history of the forecasts

**Source**, the [Previous Runs docs](https://open-meteo.com/en/docs/previous-runs-api): "Most models are archived from January 2024" and "JMA GSM and MSM — available from 2018".

**Step: request one month** the way the loader would: all nine points, all three models, every variable at three leads (`latest`, `previous_day1`, `previous_day2`; the first two are needed for the lookahead-leak scenario F2). Watch the `X-RateLimit-Cost` header.

In [9]:
PREVIOUS_RUNS = "https://previous-runs-api.open-meteo.com/v1/forecast"
LEADS = ("", "_previous_day1", "_previous_day2")
points = {
    "latitude": ",".join(f"{lat}" for _, lat, _ in WEATHER_POINTS),
    "longitude": ",".join(f"{lon}" for _, _, lon in WEATHER_POINTS),
}
tokyo = {"latitude": "35.69", "longitude": "139.69"}


def previous_runs(name: str, start: str, end: str, models, variables, pts=points):
    params = {
        **pts,
        "hourly": ",".join(v + lead for v in variables for lead in LEADS),
        "models": ",".join(models),
        "start_date": start,
        "end_date": end,
    }
    raw, headers = get(
        FetchItem("openmeteo_previous_runs_probe", f"{name}.json", PREVIOUS_RUNS, params=params)
    )
    return json.loads(raw), headers


month, month_headers = previous_runs(
    "2025-06", "2025-06-01", "2025-06-30", WEATHER_MODELS, WEATHER_VARIABLES
)
print(
    "status",
    month_headers.get("x-status"),
    "| X-RateLimit-Cost:",
    month_headers.get("x-ratelimit-cost"),
)
print(
    "response:", f"{(CACHE / 'openmeteo_previous_runs_probe__2025-06.json').stat().st_size:,} bytes"
)

status 200 | X-RateLimit-Cost: 104.142845
response: 1,743,729 bytes


In [10]:
first_point = month[0] if isinstance(month, list) else month
hourly_df = pd.DataFrame(first_point["hourly"])
coverage = {}
for column in hourly_df.columns.drop("time"):
    model = next(m for m in WEATHER_MODELS if column.endswith("_" + m))
    base = column.removesuffix("_" + model)
    lead = next((lead for lead in ("_previous_day1", "_previous_day2") if base.endswith(lead)), "")
    coverage[(base.removesuffix(lead), lead.lstrip("_") or "latest", model)] = (
        hourly_df[column].notna().mean()
    )
pd.Series(coverage).unstack([2, 1]).round(2)

jma_msm                             jma_gsm                \
                     latest previous_day1 previous_day2  latest previous_day1   
cloud_cover             1.0           1.0           1.0     1.0           1.0   
diffuse_radiation       1.0           1.0           1.0     0.0           0.0   
direct_radiation        1.0           1.0           1.0     0.0           0.0   
shortwave_radiation     1.0           1.0           1.0     0.0           0.0   
temperature_2m          1.0           1.0           1.0     1.0           1.0   
wind_speed_10m          1.0           1.0           1.0     1.0           1.0   

                                  ecmwf_ifs025                              
                    previous_day2       latest previous_day1 previous_day2  
cloud_cover                   1.0          1.0           1.0           1.0  
diffuse_radiation             0.0          1.0           1.0           1.0  
direct_radiation              0.0          1.0           1.0           1.0  
shortwave_radiation           0.0          1.0           1.0           1.0  
temperature_2m                1.0          1.0           1.0           1.0  
wind_speed_10m                1.0          1.0           1.0           1.0

**What it shows.** **Observed:** one month costs about **104 calls**, not 1, against a free limit of 600 a minute and 10,000 a day. GSM has no radiation at any lead; everything else is complete.

**Step: understand the cost.** A formula that fits: points × max(1, columns ÷ 10) × max(1, days ÷ 14), where columns counts every variable × lead × model.

In [11]:
def open_meteo_cost(points: int, columns: int, days: float) -> float:
    return points * max(1, columns / 10) * max(1, days / 14)


predicted = open_meteo_cost(9, len(WEATHER_VARIABLES) * len(LEADS) * len(WEATHER_MODELS), 30)
print(f"predicted {predicted:.6f} vs header {month_headers.get('x-ratelimit-cost')}")

predicted 104.142857 vs header 104.142845


**Observed:** the formula matches the header to six decimals. Splitting a month into two-week chunks or splitting models into separate requests doesn't make the total cheaper; it only changes how the cost is spread.

**Step: check how far back each model's leads really go.** The docs promise JMA from 2018 and most models from January 2024. Probe one day at a time, at one point. A trap found on the way: **when a request names a single model, Open-Meteo drops the model suffix** from the column names, so the lookup must try both forms.

In [12]:
def lead_history(model: str, dates: list[str]) -> list[dict]:
    rows = []
    for day in dates:
        data, _ = previous_runs(
            f"scan_{model}_{day}", day, day, (model,), ("shortwave_radiation",), tokyo
        )
        hourly = data.get("hourly", {})
        # With a single model, Open-Meteo drops the model suffix from column names.
        values = hourly.get(f"shortwave_radiation_previous_day2_{model}") or hourly.get(
            "shortwave_radiation_previous_day2", []
        )
        share = sum(v is not None for v in values) / max(len(values), 1)
        rows.append({"model": model, "date": day, "previous_day2 non-null": share})
    return rows


scan = pd.DataFrame(
    lead_history(
        "jma_msm",
        ["2019-06-01", "2020-06-01", "2021-06-01", "2022-06-01", "2023-06-01", "2023-12-01"],
    )
    + lead_history(
        "ecmwf_ifs025", ["2024-01-01", "2024-03-01", "2024-05-01", "2024-07-01", "2025-01-01"]
    )
)
scan.pivot(index="date", columns="model", values="previous_day2 non-null")

model,ecmwf_ifs025,jma_msm
date,,
2019-06-01,NaN,0.0
2020-06-01,NaN,0.0
2021-06-01,NaN,0.0
2022-06-01,NaN,0.0
2023-06-01,NaN,1.0
2023-12-01,NaN,1.0
2024-01-01,0.0,NaN
2024-03-01,0.0,NaN
2024-05-01,1.0,NaN


In [13]:
data_2018, _ = previous_runs(
    "jma_2018", "2018-06-01", "2018-06-02", ("jma_msm", "jma_gsm"), ("cloud_cover",), tokyo
)
{
    k: f"{sum(v is not None for v in vals)}/{len(vals)}"
    for k, vals in data_2018["hourly"].items()
    if k != "time"
}

{'shortwave_radiation_jma_msm': '48/48',
 'shortwave_radiation_previous_day1_jma_msm': '0/48',
 'shortwave_radiation_previous_day2_jma_msm': '0/48',
 'cloud_cover_jma_msm': '48/48',
 'cloud_cover_previous_day1_jma_msm': '0/48',
 'cloud_cover_previous_day2_jma_msm': '0/48',
 'shortwave_radiation_jma_gsm': '0/48',
 'shortwave_radiation_previous_day1_jma_gsm': '0/48',
 'shortwave_radiation_previous_day2_jma_gsm': '0/48',
 'cloud_cover_jma_gsm': '48/48',
 'cloud_cover_previous_day1_jma_gsm': '48/48',
 'cloud_cover_previous_day2_jma_gsm': '48/48'}

**What it shows.** **Observed:**

- **MSM's leads start between mid-2022 and mid-2023**, not 2018. In 2018 MSM has only the latest value, while GSM's leads go back that far
- **ECMWF's leads start between March and May 2024**, not January 2024. The first months of our window have no ECMWF fallback
- Both still cover our training window, which starts in February 2024 with the 30-minute TEPCO labels

> **Learned → design**
>
> - **History from 2024-01 to now, one request per calendar month**, all models and all three leads: about 35 requests and 3,600 calls, inside one day's quota. Paced so no minute exceeds 600 calls, which at about 104 calls a request means one request every 12 s or so; about 8 minutes in all
> - **The loader reads `X-RateLimit-Cost`** and paces itself; if the day's quota runs low it stops cleanly and resumes the next day
> - **Parsers accept both column forms**, with and without the model suffix
> - **Missing leads are data, not errors.** GSM radiation is always empty, ECMWF before spring 2024 is empty; DWD keeps the nulls and features fall back explicitly

## 4. Open-Meteo Single Runs: exactly what was available at decision time

`previous_day2` is safe but older than what a forecaster really had at 09:00. Single runs give whole runs by start time. **Source**, the [Single Runs docs](https://open-meteo.com/en/docs/single-runs-api): "Archival runs are available from 2nd of April 2026 for most models", and runs become available "typically 4–6 hours for global models … and 1–3 hours for regional models" after they start.

**Step: request the candidate runs** for one day and see what each covers.

In [14]:
SINGLE_RUNS = "https://single-runs-api.open-meteo.com/v1/forecast"
run_day = (now_jst - timedelta(days=2)).date()
RUNS = [
    ("jma_msm", 12),
    ("jma_msm", 18),
    ("jma_msm", 21),
    ("jma_gsm", 12),
    ("jma_gsm", 18),
    ("ecmwf_ifs025", 12),
    ("ecmwf_ifs025", 18),
]
rows = []
for model, hour in RUNS:
    run = f"{run_day}T{hour:02d}:00"
    params = {**points, "hourly": "cloud_cover,shortwave_radiation", "models": model, "run": run}
    raw, headers = get(
        FetchItem("openmeteo_single_runs_probe", f"{model}_{run}.json", SINGLE_RUNS, params=params)
    )
    data = json.loads(raw)
    first = data[0] if isinstance(data, list) else data
    row = {
        "model": model,
        "run (UTC)": pd.Timestamp(run, tz="UTC"),
        "cost": headers.get("x-ratelimit-cost"),
    }
    if "hourly" in first:
        h = pd.DataFrame(first["hourly"])
        h["time"] = pd.to_datetime(h["time"]).dt.tz_localize("UTC")
        covered = h.dropna(subset=["cloud_cover"])["time"]
        row |= {
            "covers to": covered.max(),
            "horizon (h)": int((covered.max() - row["run (UTC)"]).total_seconds() // 3600),
            "radiation": "yes" if h["shortwave_radiation"].notna().any() else "no",
        }
    else:
        row["error"] = first.get("reason")
    rows.append(row)
single = pd.DataFrame(rows)
single.assign(**{"covers to": single["covers to"].dt.tz_convert("Asia/Tokyo")})

,model,run (UTC),cost,error,covers to,horizon (h),radiation
0,jma_msm,2026-11-13 12:00:00+00:00,1.0,The requested model run is not available. Mode...,NaT,NaN,NaN
1,jma_msm,2026-11-13 18:00:00+00:00,9.0,NaN,2026-11-15 18:00:00+09:00,39.0,yes
2,jma_msm,2026-11-13 21:00:00+00:00,9.0,NaN,2026-11-15 21:00:00+09:00,39.0,yes
3,jma_gsm,2026-11-13 12:00:00+00:00,9.0,NaN,2026-11-20 20:00:00+09:00,167.0,no
4,jma_gsm,2026-11-13 18:00:00+00:00,9.0,NaN,2026-11-19 20:00:00+09:00,137.0,no
5,ecmwf_ifs025,2026-11-13 12:00:00+00:00,9.0,NaN,2026-11-20 20:00:00+09:00,167.0,yes
6,ecmwf_ifs025,2026-11-13 18:00:00+00:00,9.0,NaN,2026-11-20 05:00:00+09:00,146.0,yes


**What it shows.** **Observed:** not every run is archived (MSM's 12 UTC run returns "not available"). MSM's 18 and 21 UTC runs reach 39 hours ahead; GSM and ECMWF reach about a week. Each request costs 9 calls: one per point.

**Step: which run did a forecaster at 09:00 JST really have?** Add each model's availability delay to the run start, compare with `T`, and check the run still covers the target day's daylight.

In [15]:
DELAY = {
    "jma_msm": pd.Timedelta(hours=3),
    "jma_gsm": pd.Timedelta(hours=6),
    "ecmwf_ifs025": pd.Timedelta(hours=6),
}
T = pd.Timestamp(f"{run_day + timedelta(days=1)} 09:00", tz="Asia/Tokyo")
D = (T + pd.Timedelta(days=1)).normalize()
daylight_end = D + pd.Timedelta(hours=19)
ok = single.dropna(subset=["covers to"]).copy()
ok["available at (JST)"] = (ok["run (UTC)"] + ok["model"].map(DELAY)).dt.tz_convert("Asia/Tokyo")
ok["known at T?"] = ok["available at (JST)"] <= T
ok["covers D until 19:00?"] = ok["covers to"] >= daylight_end
print(f"T = {T:%Y-%m-%d %H:%M} JST, target day D = {D:%Y-%m-%d}")
ok[["model", "run (UTC)", "available at (JST)", "known at T?", "covers D until 19:00?"]]

T = 2026-11-14 09:00 JST, target day D = 2026-11-15


,model,run (UTC),available at (JST),known at T?,covers D until 19:00?
1,jma_msm,2026-11-13 18:00:00+00:00,2026-11-14 06:00:00+09:00,True,False
2,jma_msm,2026-11-13 21:00:00+00:00,2026-11-14 09:00:00+09:00,True,True
3,jma_gsm,2026-11-13 12:00:00+00:00,2026-11-14 03:00:00+09:00,True,True
4,jma_gsm,2026-11-13 18:00:00+00:00,2026-11-14 09:00:00+09:00,True,True
5,ecmwf_ifs025,2026-11-13 12:00:00+00:00,2026-11-14 03:00:00+09:00,True,True
6,ecmwf_ifs025,2026-11-13 18:00:00+00:00,2026-11-14 09:00:00+09:00,True,True


**What it shows.** **Observed**, with the assumed delays: MSM's 21 UTC run becomes available exactly at `T`, too close to trust; its 18 UTC run is safely available and covers the target day only until about 18:00 JST, which in summer clips the last hour of sunlight. For GSM and ECMWF, the 18 UTC run also lands right at `T`, and the 12 UTC run is safe.

> **Learned → design**
>
> - **Fetch two runs per model per day** from 2 April 2026: MSM 18 and 21 UTC, GSM and ECMWF 12 and 18 UTC. The as-of rule then picks the newest one that was available at `T`; the safer one is always there as a fallback
> - **About 6 runs × 227 days ≈ 1,400 requests × 9 calls ≈ 12,000 calls**: more than one day's free quota, so this load spans two days. Each request asks for all six variables, still within the 10-variable band, so the cost stays 9
> - **The assumed delays decide which run counts.** The live 08:45 captures record which MSM run was really out by then; comparing them with the archive will replace the 3-hour assumption with a measured one

## 5. The whole load

In [16]:
months = pd.period_range("2024-01", now_jst.strftime("%Y-%m"), freq="M")
single_days = (now_jst.date() - date(2026, 4, 2)).days
monthly_files = len(tepco_files[tepco_files["page"] == "monthly"])
yearly_files = len(tepco_files[tepco_files["page"] == "yearly"])
prev_cost = open_meteo_cost(9, len(WEATHER_VARIABLES) * len(LEADS) * len(WEATHER_MODELS), 30)
plan = pd.DataFrame(
    [
        ("tepco_actuals", "monthly + yearly files", monthly_files + yearly_files, 8, 0),
        ("jepx_spot", f"FY2016–FY{fy_now - 1}", fy_now - 2016, 20, 0),
        (
            "openmeteo_previous_runs",
            "one request per month from 2024-01",
            len(months),
            1.7 * len(months),
            prev_cost * len(months),
        ),
        (
            "openmeteo_single_runs",
            "6 runs a day from 2026-04-02",
            6 * single_days,
            0.1 * 6 * single_days,
            9 * 6 * single_days,
        ),
    ],
    columns=["source", "what", "requests", "≈ MB", "Open-Meteo calls"],
)
plan["≈ days of free quota"] = (plan["Open-Meteo calls"] / 10_000).round(1)
plan.round({"≈ MB": 0, "Open-Meteo calls": 0})

,source,what,requests,≈ MB,Open-Meteo calls,≈ days of free quota
0,tepco_actuals,monthly + yearly files,42,8.0,0.0,0.0
1,jepx_spot,FY2016–FY2025,10,20.0,0.0,0.0
2,openmeteo_previous_runs,one request per month from 2024-01,35,60.0,3645.0,0.4
3,openmeteo_single_runs,6 runs a day from 2026-04-02,1362,136.0,12258.0,1.2


**Observed:** TEPCO and JEPX are small and quick. Open-Meteo dominates: the previous-runs load fits in one day of quota, the single-runs load needs about two. Everything together is about 1,450 requests and roughly 220 MB, spread over two days.

## 6. Confirmation probe: can the monthly re-capture skip unchanged files?

The monthly re-capture should only download files TEPCO actually rewrote. HTTP offers two ways to ask "has this changed?": `If-None-Match` with the file's `ETag`, and `If-Modified-Since` with its `Last-Modified`. An unchanged file should come back as `304 Not Modified` with no body. **Step:** try both against files whose headers we already have.

In [17]:
results_path = CACHE / "conditional_requests.json"
if results_path.exists():
    results = json.loads(results_path.read_text())
else:
    results = []
    for name in ["eria_jukyu_202610_03.csv", "eria_jukyu_202602_03.csv", "area-2023.csv"]:
        _, headers = tepco_csv(name)
        url = tepco_files.loc[tepco_files["file"] == name, "url"].iloc[0]
        for header, value in (
            ("If-None-Match", headers["etag"]),
            ("If-Modified-Since", headers["last-modified"]),
        ):
            time.sleep(2)
            r = client.get(url, headers={header: value})
            results.append(
                {
                    "file": name,
                    "asked with": header,
                    "status": r.status_code,
                    "bytes": len(r.content),
                }
            )
    results_path.write_text(json.dumps(results))
pd.DataFrame(results)

,file,asked with,status,bytes
0,eria_jukyu_202610_03.csv,If-None-Match,200,136134
1,eria_jukyu_202610_03.csv,If-Modified-Since,304,0
2,eria_jukyu_202602_03.csv,If-None-Match,200,116020
3,eria_jukyu_202602_03.csv,If-Modified-Since,304,0
4,area-2023.csv,If-None-Match,200,434359
5,area-2023.csv,If-Modified-Since,304,0


**What it shows.** **Observed:** `If-Modified-Since` works: unchanged files come back as `304` with no body. `If-None-Match` is ignored: the full file comes back every time, probably because the CDN in front of TEPCO's site doesn't honour its own ETags.

> **Learned → design**
>
> - **The monthly re-capture sends `If-Modified-Since`** with the `Last-Modified` stored in the newest sidecar for that file, never the ETag. A `304` stores nothing; a `200` stores a new version
> - **The daily capture always downloads.** Its job is to record what was there at 08:45 every day, and Dagster's freshness checks expect a file per day

## 7. What the first real load taught us

The probe sampled; the initial load in prod requested everything. Three things only showed up at full scale:

- **Single runs answered "not available" for about 40 runs per JMA run type.** The docs promise archived runs from 2 April 2026 for most models
- **The capture job's service account couldn't list the raw bucket.** `roles/storage.objectCreator` can create objects but not list or read them, and a resumable load needs both
- **Every Cloud Run retry re-requested the missing runs.** A missing run counted as a failure, a failure made the task retry, and each retry spent about 1,500 quota calls on answers that can't change

**Step: confirm where JMA's archive really starts.** One point and one variable per request, so each costs about one call.

In [18]:
rows = []
for model, hour in (
    ("jma_msm", 18),
    ("jma_msm", 21),
    ("jma_gsm", 12),
    ("jma_gsm", 18),
    ("ecmwf_ifs025", 12),
):
    for day in ("2026-04-15", "2026-05-11", "2026-05-12", "2026-05-29", "2026-06-11"):
        run = f"{day}T{hour:02d}:00"
        params = {**tokyo, "hourly": "cloud_cover", "models": model, "run": run}
        raw, _ = get(
            FetchItem(
                "openmeteo_single_runs_scan", f"{model}_{run}.json", SINGLE_RUNS, params=params
            )
        )
        data = json.loads(raw)
        rows.append({"run": f"{model} {hour:02d}Z", "day": day, "archived": "error" not in data})
pd.DataFrame(rows).pivot(index="run", columns="day", values="archived")

day,2026-04-15,2026-05-11,2026-05-12,2026-05-29,2026-06-11
run,,,,,
ecmwf_ifs025 12Z,True,True,True,True,True
jma_gsm 12Z,False,False,True,True,True
jma_gsm 18Z,False,False,True,True,True
jma_msm 18Z,False,False,True,True,True
jma_msm 21Z,False,False,True,True,True


**What it shows.** **Observed:** ECMWF's runs are there from April. Every JMA run type checked is missing on 11 May and present from **12 May 2026**. After that, scattered runs are still missing: the first load met gaps into June.

> **Learned → design**
>
> - **A run the archive doesn't have is a recorded gap, not a failure.** The loader stores Open-Meteo's answer with its HTTP status in the sidecar; the resumed load skips it like any stored file, so it is requested once, ever. Parsers read `gap: true` from the sidecar
> - **JMA's exact-run fidelity starts on 12 May 2026.** Before that, JMA history is `previous_day2` only
> - **The capture account gets `roles/storage.objectViewer` as well.** Listing and reading don't weaken create-only: neither role can delete or overwrite
> - **A source should only fail for errors a retry can fix.** Otherwise Cloud Run's task retries multiply the cost of something permanent

## Learned improvements

Everything this notebook and the first load changed in the design, with where the evidence is:

| Learned | Evidence | Change |
| --- | --- | --- |
| TEPCO rewrites a month's file for weeks after it ends | Section 1: `Last-Modified` weeks into the next month | Capture the previous month every day |
| TEPCO occasionally rewrites old files in bulk | Section 1: three files rewritten on one day in October 2024 | Monthly re-capture of every monthly file |
| TEPCO honours `If-Modified-Since` but ignores `If-None-Match` | Section 6: `304` vs a full `200` | The re-capture sends the stored `Last-Modified` |
| Encoding varies from file to file | Section 1: UTF-8 and CP932 in the same series | Parsers detect the encoding per file |
| The yearly TEPCO format differs: 万kWh, two-row header, thousands separators | Section 1 | A separate parser, built when a phase needs the hourly years |
| Open-Meteo history costs by size | Section 3: `X-RateLimit-Cost`, matched by a formula | Pacing from the cost header and a call budget per run |
| Single-model responses drop the model suffix | Section 3 | Parsers accept both column forms |
| Lead history is shorter than documented | Section 3: MSM from mid-2022 to mid-2023, ECMWF from spring 2024 | Nulls are kept; features fall back explicitly |
| Some decision-time runs land exactly at `T` | Section 4 | Two runs per model per day |
| JMA's single-run archive starts on 12 May 2026, with scattered holes after | Section 7 and the first load | Missing runs stored once as recorded gaps |
| `objectCreator` can't list or read | The first load: `403` on listing | Add `objectViewer` to the capture account |
| Permanent errors multiplied by retries | The first load: about 1,500 calls per retry | Only retryable errors fail a source |

## Where the sources and the data disagree

| Topic | Source says | Observed | What we do |
| --- | --- | --- | --- |
| JMA history | "JMA GSM and MSM — available from 2018" ([Previous Runs docs](https://open-meteo.com/en/docs/previous-runs-api)) | GSM's leads go back to 2018; MSM has only the latest value until mid-2022 to mid-2023 | Doesn't affect phase 1, which starts in 2024; matters if the hourly years are ever used |
| Other models' history | "Most models are archived from January 2024" | ECMWF's leads start between March and May 2024 | No ECMWF fallback for the first months; DWD keeps the gap as nulls |
| Archived single runs | Available from 2 April 2026 ([Single Runs docs](https://open-meteo.com/en/docs/single-runs-api)) | Not for JMA: JMA's runs start on 12 May 2026, with scattered later holes; MSM 12 UTC is never archived | Store a missing run as a recorded gap, requested once; JMA's exact-run period starts on 12 May |
| TEPCO corrections | Past values may be corrected ([actuals page](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html)) | Files rewritten weeks after the month, and in bulk months later | Capture the previous month daily, re-capture every month monthly |
| TEPCO encoding | Not stated | UTF-8 or CP932, changing from file to file | Detect per file |
| TEPCO caching | Files carry both `ETag` and `Last-Modified` | Only `If-Modified-Since` is honoured; `If-None-Match` returns the full file | Conditional requests use `Last-Modified` |

## Sources

| Page | Supports |
| --- | --- |
| [TEPCO actuals, monthly files](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html) | File list from February 2024; retroactive corrections |
| [TEPCO actuals, yearly files](https://www.tepco.co.jp/forecast/html/area_jukyu_p-j.html) | File list FY2016–FY2023 |
| [JEPX spot market page](https://www.jepx.jp/electricpower/market-data/spot/) | Yearly price files by form POST |
| [Open-Meteo Previous Runs API](https://open-meteo.com/en/docs/previous-runs-api) | Lead variables; stated archive depth |
| [Open-Meteo Single Runs API](https://open-meteo.com/en/docs/single-runs-api) | Archive from 2 April 2026; availability delays |
| [Open-Meteo terms](https://open-meteo.com/en/terms) | Free limits: 600 calls a minute, 5,000 an hour, 10,000 a day |

## From probe to step-2 design

| Finding | Design |
| --- | --- |
| History is a known, finite list of files and requests | `jtf fetch --source S --initial` generates the history items for a source from name patterns and date ranges |
| Weather history has its own shapes | Two history-only sources: `openmeteo_previous_runs` (one item per month) and `openmeteo_single_runs` (one item per run) |
| Open-Meteo history is expensive and spans two days | Pacing from `X-RateLimit-Cost`, a daily quota guard, and a clean stop |
| A load may stop half-way | Resumable: an item already in the raw bucket under that source is skipped |
| Only capture jobs may write raw files | Runs as one-off executions of the capture jobs with `--initial` and a longer task timeout |
| History is labelled, not special | Raw files stored like daily ones, sidecar `load_type: initial`; `known_at` comes from the history rules at parse time |
| Encodings, two TEPCO formats, column suffixes, null leads | Parser requirements for step 5, recorded now |
| TEPCO corrections arrive late | Daily capture of the previous month; a monthly re-capture of every monthly file, with `If-Modified-Since` |

## Decisions

| Decision | Decided |
| --- | --- |
| Previous-month capture window | Every day, not only days 1–5 |
| Monthly re-capture of all monthly TEPCO files | Yes, on the 1st of each month, with `If-Modified-Since` so only rewritten files are stored |
| History start for weather | 2024-01 for previous runs (one month before the labels); 2026-04-02 for single runs. Revisit if the backtest needs more |
| JEPX history start | FY2016 |
| Hourly TEPCO years | Load raw now; explore and build their parser only when a later phase needs them |